In [1]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist

In [13]:
# Define function that does everything
def match_nuclei_datasets(dataset_Cellpose_path, dataset_ScanR_path, tolerance=10, 
                          x_col_cellpose='nucleus_centroid_x', y_col_cellpose='nucleus_centroid_y',
                          x_col_ScanR='nucleus_centroid_x', y_col_ScanR='nucleus_centroid_y'):
    """
    Match nuclei between two datasets based on centroid proximity.
    
    Parameters:
    -----------
    dataset1_path : str
        Path to first CSV file (e.g., your program's output)
    dataset2_path : str
        Path to second CSV file (e.g., other program's output)
    tolerance : float
        Maximum distance (in pixels) for nuclei to be considered a match
    x_col_cellpose, y_col_cellpose : str
        Column names for x,y coordinates in dataset 1
    x_col_ScanR, y_col_ScanR : str
        Column names for x,y coordinates in dataset 2
    
    Returns:
    --------
    pd.DataFrame
        Combined DataFrame with matched and unmatched nuclei from both datasets
    """
    
    # Load both datasets
    df1 = pd.read_csv(dataset_Cellpose_path, skiprows=1)
    df2 = pd.read_csv(dataset_ScanR_path, sep=';')
    
    # Add source labels
    df1['source'] = 'Cellpose'
    df2['source'] = 'ScanR'
    
    # Extract coordinates
    coords_Cellpose = df1[[y_col_cellpose, x_col_cellpose]].values
    coords_ScanR = df2[[y_col_ScanR, x_col_ScanR]].values
    
    # Calculate pairwise distances between all nuclei
    distances = cdist(coords_Cellpose, coords_ScanR)
    
    # Find matches within tolerance
    matched_pairs = []
    matched_idx1 = set()
    matched_idx2 = set()
    
    # For each nucleus in dataset1, find closest match in dataset2
    for i in range(len(coords_Cellpose)):
        # Finds the closest nucleus in ScanR to the one selected with i in the cellpose dataset
        min_dist_idx = np.argmin(distances[i, :])
        # min_dist is that smallest value between the two datasets for i in cellpose
        min_dist = distances[i, min_dist_idx]

        # Setting the requirement of the distance having to be within tolerance
        if min_dist <= tolerance:
            # Check if this is also the closest match from the ScanR dataset's perspective
            if np.argmin(distances[:, min_dist_idx]) == i:
                matched_pairs.append({
                    'idx1': i,
                    'idx2': min_dist_idx,
                    'distance': min_dist
                })
                # Marking the nuclei as mached in both datasets
                matched_idx1.add(i)
                matched_idx2.add(min_dist_idx)
    
    # Create combined dataframe
    result_rows = []
    
    # Add CONFIRMED nuclei (matched between both datasets)
    for match in matched_pairs:
        row1 = df1.iloc[match['idx1']].to_dict()
        row2 = df2.iloc[match['idx2']].to_dict()

        # Creates a combined dictionary by adding the source to the respective information from the two datasets
        combined_row = {
            'status': 'CONFIRMED',
            'match_distance': match['distance'],
            'source': 'Both',
            # Dataset 1 data (prefix with d_Cellpose_), skipping the source, because it is already in the dataset with the tag 'Both'
            **{f'd_Cellpose_{k}': v for k, v in row1.items() if k != 'source'},
            # Dataset 2 data (prefix with d_ScanR_)
            **{f'd_ScanR_{k}': v for k, v in row2.items() if k != 'source'}
        }
        # Adding the Data to a combined dictionary
        result_rows.append(combined_row)
    
    # Add UNCONFIRMED nuclei from the Cellpose Dataset
    unmatched1 = [i for i in range(len(df1)) if i not in matched_idx1]
    for i in unmatched1:
        row = df1.iloc[i].to_dict()
        combined_row = {
            'status': 'UNCONFIRMED',
            'match_distance': np.nan,
            'source': 'Cellpose_only',
            **{f'd_Cellpose_{k}': v for k, v in row.items() if k != 'source'},
        }
        result_rows.append(combined_row)
    
    # Add UNCONFIRMED nuclei from Dataset 2
    unmatched2 = [i for i in range(len(df2)) if i not in matched_idx2]
    for i in unmatched2:
        row = df2.iloc[i].to_dict()
        combined_row = {
            'status': 'UNCONFIRMED',
            'match_distance': np.nan,
            'source': 'Dataset2_only',
            **{f'd_ScanR_{k}': v for k, v in row.items() if k != 'source'},
        }
        result_rows.append(combined_row)
    
    # Create final DataFrame
    result_df = pd.DataFrame(result_rows)
    
    # Print summary statistics
    print("\nData_Summary")
    print("\n------------")
    print(f"Total nuclei in Dataset1: {len(df1)}")
    print(f"Total nuclei in Dataset2: {len(df2)}")
    print(f"Confirmed matches: {len(matched_pairs)}")
    print(f"Unmatched in Dataset1: {len(unmatched1)}")
    print(f"Unmatched in Dataset2: {len(unmatched2)}")
    print(f"Total rows in result: {len(result_df)}")
    
    return result_df


In [27]:

# Call on function with the specific parameters
matched_df = match_nuclei_datasets(
    dataset_Cellpose_path=r'Y:\Group Members\Valentin Aubry\01_Data\Data_Andreas_single\ATR2_24h--W00044--P00021--Z00000--T00000----nuclei_DF_SEM.csv',
    dataset_ScanR_path=r'Y:\Group Members\Valentin Aubry\01_Data\Test_Data_Andreas\AP464_U2OS_gH2AX(488)_53BP1(568)_EdU(647)_20x_002_20251012_Position_check_all_Main_Well44_position21.csv',
    tolerance=30,
    x_col_cellpose='centr_x',  # Adjust to your column names
    y_col_cellpose='centr_y',
    x_col_ScanR='Center X ',  # Adjust to other program's column names
    y_col_ScanR='Center Y '
)





Data_Summary

------------
Total nuclei in Dataset1: 112
Total nuclei in Dataset2: 122
Confirmed matches: 109
Unmatched in Dataset1: 3
Unmatched in Dataset2: 13
Total rows in result: 125


In [28]:
# Save results
matched_df.to_csv(r'Y:\Group Members\Valentin Aubry\01_Data\Data_Andreas_single\Nuclei_Comparison_vs_ScanR.csv', index=False)

In [30]:
filename = r'Y:\Group Members\Valentin Aubry\01_Data\Data_Andreas_single\Nuclei_Comparison_vs_ScanR.csv'

# Write sep=, line for Excel compatibility
with open(filename, 'w') as f:
    f.write('sep=,\n')

# Write data with header
matched_df.to_csv(filename, mode='a', header=True, index=False)